In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import numpy as np

# ── Connection ─────────────────────────────────────────────
conn = psycopg2.connect(
    host     = "localhost",
    database = "upi_intelligence",
    user     = "postgres",
    password = "NewStrongPassword@123",
    port     = 5432
)
cursor = conn.cursor()
print("✅ Connected to PostgreSQL")

# ── Paths ──────────────────────────────────────────────────
cleaned_path = r"D:\Projects\End-to-end projects\10. UPI Payments Intelligence\Data\Cleaned"
raw_path     = r"D:\Projects\End-to-end projects\10. UPI Payments Intelligence\Data\Raw"

# ══════════════════════════════════════════════════════════
# LOAD MERCHANTS
# ══════════════════════════════════════════════════════════
df_merchants = pd.read_csv(raw_path + r"\upi_merchants.csv")
df_merchants = df_merchants.where(pd.notnull(df_merchants), None)

merchant_cols = ['merchant_id', 'merchant_name', 'merchant_category',
                 'merchant_city', 'merchant_city_tier', 'is_active']

cursor.execute("TRUNCATE TABLE merchants;")
conn.commit()

values = [tuple(row) for row in df_merchants[merchant_cols].values]
execute_values(cursor, f"""
    INSERT INTO merchants ({', '.join(merchant_cols)})
    VALUES %s ON CONFLICT (merchant_id) DO NOTHING
""", values)
conn.commit()
print(f"✅ Merchants inserted: {len(df_merchants):,} rows")

# ══════════════════════════════════════════════════════════
# LOAD USERS
# ══════════════════════════════════════════════════════════
df_users = pd.read_csv(cleaned_path + r"\users_segmented.csv",
                       parse_dates=['registration_date'])
df_users = df_users.where(pd.notnull(df_users), None)

user_cols = [
    'user_id', 'age', 'gender', 'city', 'city_tier',
    'preferred_platform', 'activity_level', 'monthly_txn_frequency',
    'device_type', 'registration_date', 'segment', 'segment_name',
    'avg_txn_amount', 'total_spend', 'total_txns',
    'platform_loyalty_rate', 'avg_spend_per_day'
]

cursor.execute("TRUNCATE TABLE users;")
conn.commit()

values = [tuple(row) for row in df_users[user_cols].values]
execute_values(cursor, f"""
    INSERT INTO users ({', '.join(user_cols)})
    VALUES %s ON CONFLICT (user_id) DO NOTHING
""", values)
conn.commit()
print(f"✅ Users inserted: {len(df_users):,} rows")

# ══════════════════════════════════════════════════════════
# LOAD TRANSACTIONS
# ══════════════════════════════════════════════════════════
df_txn = pd.read_csv(cleaned_path + r"\cleaned_transactions.csv",
                     parse_dates=['txn_datetime', 'txn_date'])
df_txn = df_txn.where(pd.notnull(df_txn), None)

txn_cols = [
    'txn_id', 'txn_datetime', 'txn_date', 'txn_month', 'txn_year',
    'txn_hour', 'day_of_week', 'user_id', 'user_city', 'user_city_tier',
    'user_age', 'user_gender', 'user_activity_level', 'merchant_id',
    'merchant_name', 'merchant_category', 'merchant_city', 'platform_used',
    'preferred_platform', 'is_preferred_platform', 'payment_mode',
    'device_type', 'amount', 'status', 'unusual_hour_flag',
    'high_amount_flag', 'round_amount_flag', 'velocity_flag',
    'fraud_score', 'is_fraud', 'is_weekend', 'time_of_day',
    'is_festival_month', 'quarter', 'is_success', 'is_failed',
    'is_cross_city', 'age_group'
]

cursor.execute("TRUNCATE TABLE transactions;")
conn.commit()

batch_size = 10000
total_rows = len(df_txn)
inserted   = 0

for i in range(0, total_rows, batch_size):
    batch  = df_txn[txn_cols].iloc[i:i+batch_size]
    values = [tuple(row) for row in batch.values]
    execute_values(cursor, f"""
        INSERT INTO transactions ({', '.join(txn_cols)})
        VALUES %s ON CONFLICT (txn_id) DO NOTHING
    """, values)
    conn.commit()
    inserted += len(batch)
    if inserted % 50000 == 0:
        print(f"  Transactions: {inserted:,} / {total_rows:,} inserted...")

print(f"✅ Transactions inserted: {inserted:,} rows")

# ══════════════════════════════════════════════════════════
# VERIFY
# ══════════════════════════════════════════════════════════
cursor.execute("""
    SELECT 'transactions' AS tbl, COUNT(*) FROM transactions
    UNION ALL
    SELECT 'users',        COUNT(*) FROM users
    UNION ALL
    SELECT 'merchants',    COUNT(*) FROM merchants
""")
rows = cursor.fetchall()
print(f"\n{'='*40}")
print(f"FINAL ROW COUNTS")
print(f"{'='*40}")
for row in rows:
    print(f"  {row[0]:<15} {row[1]:>10,} rows")

cursor.close()
conn.close()
print(f"\n✅ Connection closed")

✅ Connected to PostgreSQL
✅ Merchants inserted: 2,000 rows
✅ Users inserted: 10,000 rows
  Transactions: 50,000 / 500,000 inserted...
  Transactions: 100,000 / 500,000 inserted...
  Transactions: 150,000 / 500,000 inserted...
  Transactions: 200,000 / 500,000 inserted...
  Transactions: 250,000 / 500,000 inserted...
  Transactions: 300,000 / 500,000 inserted...
  Transactions: 350,000 / 500,000 inserted...
  Transactions: 400,000 / 500,000 inserted...
  Transactions: 450,000 / 500,000 inserted...
  Transactions: 500,000 / 500,000 inserted...
✅ Transactions inserted: 500,000 rows

FINAL ROW COUNTS
  transactions       500,000 rows
  users               10,000 rows
  merchants            2,000 rows

✅ Connection closed
